# Notebook 2: Serial communication with the Keithley 2000 multimeter

*Python for Instrumentation and Laboratory Experiments, Pr. T. Angot*

**Goal:** control the Keithley 2000 from Python: find its serial port, ask for its identity, configure it to measure DC voltages in the millivolt range, and read one value.
All the outputs below were obtained in the computer room with the real instrument.

## How the computer talks to the instrument

- The Keithley 2000 has an **RS232** port. It is connected to the computer with an RS232 → USB adapter (FTDI chip), which Windows sees as a serial port, here `COM3`.
- RS232 is an **asynchronous serial** link: each character is sent bit by bit, framed by a start bit and a stop bit. Both sides must use the same settings, chosen in the menu of the Keithley:
  **9600 baud, 8 data bits, no parity, 1 stop bit** (often written *8N1*).
- The commands are plain text in the **SCPI** language (e.g. `*IDN?`, `:READ?`). A command ending with `?` is a question: the instrument sends back an answer.
- Each command ends with a carriage return `\r` (the terminator configured on the Keithley). `pyserial` sends **bytes**, hence the `b'...'` strings, and the answers are decoded from ASCII into Python strings.
- We open the port with `with serial.Serial(...) as ser:` so that it is always closed at the end of the cell, even if an error happens (otherwise the port stays busy and the next cell cannot open it).

Import the serial library, scan for port to identify the serial com, define the com address

In [1]:
import serial

In [2]:
import serial.tools.list_ports as port_list
ports = list(port_list.comports())
for port in ports: print(port)

COM3 - USB Serial Port (COM3)


Only one serial device is connected: the USB adapter of the Keithley, on `COM3`.

In [3]:
serial_address="COM3"
print(serial_address)

COM3


## Preliminary work to test the serial communication with the Keithley2000

Ask for the instrument ID, set the insturment to Volt DC, ask that the instrument to check if it is correctly configured in this mode configured, set the integration time to 5, the range to mV <br>
Retrieve one data

### 1. Identification

In [4]:
with serial.Serial(serial_address,baudrate=9600,bytesize=8,parity='N',stopbits=1,timeout=.05) as ser:

    ser.write(b'*IDN?\r')
    l=ser.readline()
    response=l.decode('ascii',errors='replace')
    print(response)


KEITHLEY INSTRUMENTS INC.,MODEL 2000,4416105,B01  /A02  



The answer gives the manufacturer, the model (2000), the serial number of this unit (4416105) and the firmware versions. The communication works in both directions.

### 2. Select the DC voltage function and check it

In [5]:
with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=1) as ser:
    ser.write(b":CONF:VOLT:DC\r")
    ser.write(b":SENS:FUNC?\r")
    print(ser.readline().decode('ascii').strip())   # should print "VOLT:DC"

"VOLT:DC"


### 3. Integration time and range

- `NPLC` = *number of power line cycles*: the time during which the voltage is integrated, in units of one period of the mains (1/50 Hz = 20 ms). NPLC 5 means 100 ms per reading: slower, but the 50 Hz noise is averaged out.
- `RANG 0.1` selects the 100 mV range, the smallest one, adapted to a thermocouple (a type K gives about 41 µV/°C, so less than 21 mV up to 500 °C).

After sending the settings we ask them back (`?`) to check that the instrument accepted them.

In [6]:
import time

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=2) as ser:
    ser.write(b":SENS:FUNC 'VOLT:DC'\r")
    ser.write(b":SENS:VOLT:DC:NPLC 5\r")
    ser.write(b":SENS:VOLT:DC:RANG 0.1\r")
    time.sleep(0.5)

    for cmd in [b":SENS:FUNC?\r", b":SENS:VOLT:DC:NPLC?\r", b":SENS:VOLT:DC:RANG?\r"]:
        ser.reset_input_buffer()
        ser.write(cmd)
        print(cmd.decode().strip(), "->", ser.readline().decode('ascii').strip())

:SENS:FUNC? -> "VOLT:DC"
:SENS:VOLT:DC:NPLC? -> 5.00
:SENS:VOLT:DC:RANG? -> 0.100000


### 4. Read one value

`:READ?` triggers one measurement and returns it.

In [7]:
import time

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=3) as ser:
    ser.reset_input_buffer()
    ser.write(b":READ?\r")
    reading = ser.readline().decode('ascii').strip()
    print("Reading:", reading, "V")

Reading: +3.04687512E-02 V


Transform the data to a float number

In [8]:
value = float(reading)
print(value, "V")
print(value * 1000, "mV")

0.0304687512 V
30.4687512 mV


**About this value.** During the session nothing was connected to the input terminals of the Keithley (no thermocouple). On the 100 mV range the input impedance is very high (> 10 GΩ), so an open input "floats": it picks up stray charges and noise, and the reading has no physical meaning. 30 mV would correspond to about 730 °C for a type K thermocouple, which is obviously not a real temperature. The goal of this session was the communication itself, and that part works.

## Summary of the commands used

| Command | Meaning | Answer |
|---|---|---|
| `*IDN?` | identification | `KEITHLEY INSTRUMENTS INC.,MODEL 2000,...` |
| `:CONF:VOLT:DC` | configure a DC voltage measurement | none |
| `:SENS:FUNC 'VOLT:DC'` / `:SENS:FUNC?` | select / ask the measurement function | `"VOLT:DC"` |
| `:SENS:VOLT:DC:NPLC 5` / `?` | integration time = 5 power line cycles | `5.00` |
| `:SENS:VOLT:DC:RANG 0.1` / `?` | 100 mV range | `0.100000` |
| `:READ?` | trigger and return one reading (in volts) | `+3.04687512E-02` |

These functions are reused in notebook 3 to record a temperature every second.